# Bổ sung khách hàng, người bán, sản phẩm, đánh giá vào dataset đơn hàng

Task #27 (Story #4): đọc lại kết quả trung gian của Task #26 (`data/processed/orders_step1_items_payments.csv`), bổ sung thêm thông tin từ `customers`, `sellers`, `products`, `product_category_name_translation`, `order_reviews`.

Các quyết định đã chốt với User trước khi viết code:
- **Seller**: chọn seller đóng góp tổng `price` cao nhất trong đơn làm "seller chính" (phương án A), kèm cờ `items_multi_seller` để không mất hoàn toàn thông tin đa-seller (chỉ 1,30% đơn có >1 seller).
- **Product/Category**: không chọn 1 category đại diện, chỉ giữ số đếm `items_num_categories` (phương án C) — category không nằm trong danh sách đặc trưng rủi ro ở `docs/business-processes.md`, và chỉ 0,74% đơn có >1 category.
- **Review**: không chọn 1 review đại diện, tính `review_score_avg`/`min`/`max`/`review_count` sau khi dedupe dòng trùng y hệt (phương án C) — vì 37% đơn có nhiều review thực sự khác điểm nhau (202/547), không phải nhiễu do trùng lặp.

In [ ]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

step1 = pd.read_csv(PROCESSED_DIR / "orders_step1_items_payments.csv")
order_items = pd.read_csv(RAW_DIR / "olist_order_items_dataset.csv")
customers = pd.read_csv(RAW_DIR / "olist_customers_dataset.csv")
sellers = pd.read_csv(RAW_DIR / "olist_sellers_dataset.csv")
products = pd.read_csv(RAW_DIR / "olist_products_dataset.csv")
reviews = pd.read_csv(RAW_DIR / "olist_order_reviews_dataset.csv")

print(f"step1: {len(step1)} dòng, {len(step1.columns)} cột")
result = step1.copy()

C:\Users\thanh\AppData\Local\Temp\ipykernel_41080\1197945606.py:7: DtypeWarning: Columns (0: payment_has_boleto, 1: payment_has_credit_card, 2: payment_has_debit_card, 3: payment_has_not_defined, 4: payment_has_voucher) have mixed types. Specify dtype option on import or set low_memory=False.
  step1 = pd.read_csv(PROCESSED_DIR / "orders_step1_items_payments.csv")


step1: 99441 dòng, 27 cột


## Join customers

`orders.customer_id` ↔ `customers.customer_id` là quan hệ 1-1 (mỗi đơn có 1 `customer_id` riêng, xem `docs/olist-erd.md`) — join thẳng, không cần aggregate.

In [2]:
assert customers["customer_id"].is_unique, "customer_id không unique — quan hệ không còn là 1-1!"

result = result.merge(customers, on="customer_id", how="left")
assert len(result) == len(step1), "Số dòng thay đổi sau khi join customers!"
print(f"OK: {len(result)} dòng sau khi join customers.")

OK: 99441 dòng sau khi join customers.


## Seller chính theo đơn (phương án A) + cờ multi-seller

Với mỗi đơn, chọn seller có tổng `price` đóng góp cao nhất làm seller chính, lấy thêm vị trí (zip/city/state) của seller đó.

In [3]:
seller_value = order_items.groupby(["order_id", "seller_id"])["price"].sum().reset_index()
idx_max = seller_value.groupby("order_id")["price"].idxmax()
primary_seller = seller_value.loc[idx_max, ["order_id", "seller_id"]].rename(
    columns={"seller_id": "primary_seller_id"}
)

primary_seller = primary_seller.merge(
    sellers, left_on="primary_seller_id", right_on="seller_id", how="left"
).drop(columns="seller_id").rename(columns={
    "seller_zip_code_prefix": "primary_seller_zip_code_prefix",
    "seller_city": "primary_seller_city",
    "seller_state": "primary_seller_state",
})

result = result.merge(primary_seller, on="order_id", how="left")
result["items_multi_seller"] = result["items_num_sellers"] > 1

assert len(result) == len(step1), "Số dòng thay đổi sau khi join seller chính!"
print(f"OK: {len(result)} dòng. Đơn có nhiều seller (items_multi_seller=True): {result['items_multi_seller'].sum()}")

OK: 99441 dòng. Đơn có nhiều seller (items_multi_seller=True): 1278


## Số category khác nhau theo đơn (phương án C — chỉ đếm, không chọn đại diện)

In [4]:
items_with_category = order_items.merge(
    products[["product_id", "product_category_name"]], on="product_id", how="left"
)
num_categories = (
    items_with_category.groupby("order_id")["product_category_name"]
    .nunique()
    .reset_index()
    .rename(columns={"product_category_name": "items_num_categories"})
)

result = result.merge(num_categories, on="order_id", how="left")
assert len(result) == len(step1), "Số dòng thay đổi sau khi join số category!"
print(f"OK: {len(result)} dòng. Đơn có >1 category: {(result['items_num_categories'] > 1).sum()}")

OK: 99441 dòng. Đơn có >1 category: 727


## Review — dedupe rồi tính số liệu tổng hợp (phương án C)

Dedupe các dòng trùng y hệt trên (`order_id`, `review_score`, `review_creation_date`) trước khi tính trung bình, để không đếm trùng dữ liệu nhập lặp (khác với trường hợp 1 đơn thực sự có nhiều review khác điểm nhau).

In [5]:
before = len(reviews)
reviews_dedup = reviews.drop_duplicates(
    subset=["order_id", "review_score", "review_creation_date"], keep="first"
)
print(f"Loại {before - len(reviews_dedup)} dòng trùng y hệt (order_id, review_score, review_creation_date).")

review_agg = reviews_dedup.groupby("order_id").agg(
    review_score_avg=("review_score", "mean"),
    review_score_min=("review_score", "min"),
    review_score_max=("review_score", "max"),
    review_count=("review_score", "count"),
).reset_index()

result = result.merge(review_agg, on="order_id", how="left")
assert len(result) == len(step1), "Số dòng thay đổi sau khi join review!"
print(f"OK: {len(result)} dòng. Đơn có >1 review sau dedupe: {(result['review_count'] > 1).sum()}")

Loại 126 dòng trùng y hệt (order_id, review_score, review_creation_date).


OK: 99441 dòng. Đơn có >1 review sau dedupe: 422


## Kiểm tra tổng thể và lưu dataset trung gian

In [6]:
assert len(result) == len(step1), "Số dòng cuối cùng không khớp step1!"
assert result["order_id"].is_unique, "order_id không còn unique!"
print(f"OK: {len(result)} dòng, {len(result.columns)} cột, order_id vẫn unique.")

null_summary = result.isna().sum()
null_summary = null_summary[null_summary > 0]
print("\nCác cột có giá trị thiếu (NaN) và số lượng:")
print(null_summary)

OUT_PATH = PROCESSED_DIR / "orders_step2_enriched.csv"
result.to_csv(OUT_PATH, index=False)
print(f"\nĐã lưu {len(result)} dòng, {len(result.columns)} cột vào {OUT_PATH}")

OK: 99441 dòng, 41 cột, order_id vẫn unique.



Các cột có giá trị thiếu (NaN) và số lượng:
order_approved_at                  160
order_delivered_carrier_date      1783
order_delivered_customer_date     2965
items_num_items                    775
items_num_products                 775
items_num_sellers                  775
items_total_price                  775
items_total_freight                775
payment_total_value                  1
payment_num_rows                     1
payment_num_types                    1
payment_max_installments             1
payment_value_boleto                 1
payment_value_credit_card            1
payment_value_debit_card             1
payment_value_not_defined            1
payment_value_voucher                1
payment_has_boleto                   1
payment_has_credit_card              1
payment_has_debit_card               1
payment_has_not_defined              1
payment_has_voucher                  1
primary_seller_id                  775
primary_seller_zip_code_prefix     775
primary_seller_city


Đã lưu 99441 dòng, 41 cột vào ..\data\processed\orders_step2_enriched.csv
